# BYOL Downstream Classification Evaluation — LoTSS initial_pure

Evaluates BYOL encoder representations on the LoTSS initial_pure 5-class task,
using the train/test split produced by the BYOL training run.

Sections:
- **0** Configuration
- **1** Imports & Setup
- **2** Load Encoder
- **3** Load Projections (from BYOL run)
- **4** Linear Probe
- **5** Classifier Suite (KNN / RF / MLP)
- **6** Supervised Baseline Comparison
- **7** Hyperparameter Sweep Grid
- **8** Confusion Matrix (BYOL vs Supervised Baseline)
- **9** Label-Fraction Experiment

## 0. Configuration

In [1]:
CONFIG = {
    "checkpoint":    "../outputs/enb0_mlp_pd128_clos_lrconst_wd1e-4_lfull_ema0.996_vicregvar2_cov0.1_gamma0.5_f1_sw0.1_20260620_2109/byol_model_best.pt",
    "catalogue":     "../catalogues/lotss_initial_all.yaml",
    "colour_by":     "dataset",   # "dataset" | "label"
    "root":          "..",
    "baselines_dir": "../outputs/supervised_baseline_classifiers",
}
OUT_DIR = "../outputs/figures/classification"
SEED    = 42

## 1. Imports & Setup

In [2]:
import os, sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, confusion_matrix

# Add project src and scripts to path
_root = os.path.abspath("..")
for _p in [os.path.join(_root, "src"), os.path.join(_root, "scripts")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from embed_and_umap import load_encoder
from train_byol_classifiers import ALL_CLASS_NAMES, LABEL_SETS

os.makedirs(OUT_DIR, exist_ok=True)
print("Imports OK")


Imports OK


## 2. Load Encoder

Loads the BYOL checkpoint and returns the online encoder + projector. Auto-detects model type from the checkpoint config.

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

encoder, projector = load_encoder(CONFIG["checkpoint"], device)
encoder.eval()
projector.eval()

# Detect dimensions via dummy forward pass
with torch.no_grad():
    _dummy   = torch.zeros(1, 1, 89, 89).to(device)
    enc_dim  = encoder(_dummy).shape[-1]
    proj_dim = projector(encoder(_dummy)).shape[-1]

print(f"\nEncoder output dim  : {enc_dim}")
print(f"Projector output dim: {proj_dim}")


Device: cuda
Loading checkpoint: ../outputs/enb0_mlp_pd128_clos_lrconst_wd1e-4_lfull_ema0.996_vicregvar2_cov0.1_gamma0.5_f1_sw0.1_20260620_2109/byol_model_best.pt
  Model type: efficientnet-b0
  Feature compression: mlp
  Projector output dim: 128

Encoder output dim  : 1280
Projector output dim: 128


## 3. Load Projections (from BYOL run)

Loads pre-extracted projections directly from the BYOL run's `data/` directory.
This ensures the train/test split is identical to what `train_byol_classifiers.py` and the
Protege pipeline use.

`projs_cache["lotss_initial_pure"]` contains:
- `projs`     — shape `(N, proj_dim)`
- `labels`    — shape `(N,)` int64, argmax over 5 initial classes
- `split_ids` — shape `(N,)` int64, 0=train / 2=test (no val in BYOL split)


In [4]:
projs_cache = {}

_run_dir  = Path(CONFIG["checkpoint"]).parent
_data_dir = _run_dir / "data"
_byol_dir = _data_dir / "byol"

# Derive shared splits directory from the checkpoint's data_seed
_ckpt_raw   = torch.load(CONFIG["checkpoint"], map_location="cpu", weights_only=False)
_data_seed  = int(_ckpt_raw["config"]["data_seed"])
_splits_dir = _run_dir.parent / "data_splits" / str(_data_seed)

def _load_label(name):
    """Load a label .npy from splits_dir (new runs) or data_dir (old runs)."""
    p = _splits_dir / name
    return np.load(p if p.exists() else _data_dir / name)

_X_train = np.load(_byol_dir / "labelled_train_projections.npy").astype(np.float32)
_X_test  = np.load(_byol_dir / "test_projections.npy").astype(np.float32)

_lab_labels_path = _splits_dir / "labelled_train_labels.npy"
if not _lab_labels_path.exists():
    _lab_labels_path = _data_dir / "labelled_train_labels.npy"
if _lab_labels_path.exists():
    _y_train_20 = np.load(_lab_labels_path)
    if len(_y_train_20) != len(_X_train):
        # labelled_train_labels.npy was written by a different f_label run.
        # Reconstruct by concatenating labelled + unlabelled labels (covers all of
        # train_idx in order, which matches labelled_train_projections.npy for f=1).
        _unlab_lbl = _splits_dir / "unlabelled_train_labels.npy"
        if not _unlab_lbl.exists():
            _unlab_lbl = _data_dir / "unlabelled_train_labels.npy"
        if _unlab_lbl.exists():
            _y_train_20 = np.concatenate([_y_train_20, np.load(_unlab_lbl)])
        if len(_y_train_20) != len(_X_train):
            raise ValueError(f"Label count {len(_y_train_20)} != projection count {len(_X_train)}")
else:
    _all_train = _load_label("train_labels.npy")
    _lab_idx   = _load_label("labelled_train_idx.npy")
    _y_train_20 = _all_train if len(_all_train) == len(_X_train) else _all_train[_lab_idx]
_y_test_20 = _load_label("test_labels.npy")

def _initial_pure_mask(y20):
    """Rows with exactly one label among the 5 initial classes (FRI/FRII/Hybrid/Spiral/Relaxed)."""
    return y20[:, 0:5].sum(axis=1) == 1

_tr_mask = _initial_pure_mask(_y_train_20)
_te_mask = _initial_pure_mask(_y_test_20)

_X_tr_cp = _X_train[_tr_mask]
_y_tr_cp = _y_train_20[_tr_mask][:, 0:5].argmax(axis=1).astype(np.int64)
_X_te_cp = _X_test[_te_mask]
_y_te_cp = _y_test_20[_te_mask][:, 0:5].argmax(axis=1).astype(np.int64)

projs_cache["lotss_initial_pure"] = {
    "projs":     np.concatenate([_X_tr_cp, _X_te_cp]),
    "labels":    np.concatenate([_y_tr_cp, _y_te_cp]),
    "split_ids": np.concatenate([
        np.zeros(len(_X_tr_cp), dtype=np.int64),
        np.full(len(_X_te_cp), 2, dtype=np.int64),
    ]),
}
print(f"Loaded lotss_initial_pure: train={len(_X_tr_cp)}  test={len(_X_te_cp)}  proj_dim={_X_tr_cp.shape[1]}")

Loaded lotss_initial_pure: train=4690  test=2005  proj_dim=128


## 4. Linear Probe

Fit a logistic regression (`max_iter=1000`, `C=1.0`, `random_state=42`) on
train-split projections normalised with `StandardScaler` (fit on train only).
Evaluated on the fixed test split from the BYOL run.


In [5]:
def run_linear_probe(cache_entry, n_classes):
    """Fit logistic regression on train projections, report val & test metrics.

    Val is optional — skipped when split_ids contains no 1s (e.g. BYOL split).
    Returns dict with keys "test" (and "val" if present), plus "clf" and "scaler".
    """
    projs     = cache_entry["projs"]
    labels    = cache_entry["labels"]
    split_ids = cache_entry["split_ids"]

    X = {s: projs[split_ids == i]  for s, i in [("train",0),("val",1),("test",2)]}
    y = {s: labels[split_ids == i] for s, i in [("train",0),("val",1),("test",2)]}

    scaler     = StandardScaler()
    X["train"] = scaler.fit_transform(X["train"])
    if len(X["val"]):
        X["val"] = scaler.transform(X["val"])
    X["test"]  = scaler.transform(X["test"])

    clf = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
    clf.fit(X["train"], y["train"])

    results = {}
    for split in ("val", "test"):
        if len(X[split]) == 0:
            continue
        y_pred = clf.predict(X[split])
        y_prob = clf.predict_proba(X[split])
        acc    = accuracy_score(y[split], y_pred)
        f1     = f1_score(y[split], y_pred, average="macro", zero_division=0)
        rec    = recall_score(y[split], y_pred, average="macro", zero_division=0)
        if n_classes == 2:
            auc = roc_auc_score(y[split], y_prob[:, 1])
        else:
            auc = roc_auc_score(
                label_binarize(y[split], classes=list(range(n_classes))),
                y_prob, multi_class="ovr", average="macro",
            )
        results[split] = {"accuracy": acc, "f1": f1, "recall": rec, "auc": auc, "n": len(y[split])}

    results["clf"]    = clf
    results["scaler"] = scaler
    return results


In [6]:
_run_dir = Path(CONFIG["checkpoint"]).parent
_clf_dir = _run_dir / "data" / "classifiers"
_lr_json = _clf_dir / "lr_initial_pure_projections.json"

probe_results = {}
if _lr_json.exists():
    with open(_lr_json) as _f:
        _d = json.load(_f)
    probe_results["lotss_initial_pure"] = {
        "test": {
            "accuracy": _d["accuracy"],
            "f1":       _d["f1_macro"],
            "recall":   _d["recall_macro"],
            "auc":      _d["auc_macro"],
            "n":        _d["n_test"],
        },
        "clf":    None,
        "scaler": None,
    }
    r = probe_results["lotss_initial_pure"]["test"]
    print(f"Linear probe (LR) — test split:  acc={r['accuracy']:.3f}  F1={r['f1']:.3f}  AUC={r['auc']:.3f}  n={r['n']}")
else:
    print(f"No pre-computed LR results found at {_lr_json}")
    probe_results["lotss_initial_pure"] = {"test": {}, "clf": None, "scaler": None}


Linear probe (LR) — test split:  acc=0.804  F1=0.658  AUC=0.920  n=2005


## 5. Classifier Suite (KNN / RF / MLP)

Train KNN, Random Forest, and a small MLP on the frozen BYOL projections.
All classifiers use the same train/test split as the linear probe, with
`StandardScaler` normalisation fit on train only.


In [7]:
# ── MLP ───────────────────────────────────────────────────────────────────────
class _MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(512, 256), dropout=0.3):
        super().__init__()
        layers, prev = [], in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(prev, n_classes)

    def forward(self, x):
        return self.head(self.body(x))


def run_classifier_suite(cache_entry, n_classes,
                         n_neighbors=15, n_estimators=200,
                         mlp_hidden=(512, 256), mlp_epochs=100,
                         mlp_patience=15, mlp_lr=1e-3, mlp_batch=256,
                         mlp_val_frac=0.15):
    """
    Train KNN, Random Forest, and MLP on train-split BYOL projections.
    When no val split exists (BYOL split), carves mlp_val_frac from train for MLP early stopping.
    Returns dict keyed by classifier name, each with accuracy/f1/recall/auc/n.
    """
    projs     = cache_entry["projs"]
    labels    = cache_entry["labels"]
    split_ids = cache_entry["split_ids"]

    X = {s: projs[split_ids == i]  for s, i in [("train", 0), ("val", 1), ("test", 2)]}
    y = {s: labels[split_ids == i] for s, i in [("train", 0), ("val", 1), ("test", 2)]}

    scaler     = StandardScaler()
    X["train"] = scaler.fit_transform(X["train"])
    if len(X["val"]):
        X["val"] = scaler.transform(X["val"])
    X["test"]  = scaler.transform(X["test"])

    def _metrics(y_true, y_pred, y_prob):
        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
        rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
        if n_classes == 2:
            auc = roc_auc_score(y_true, y_prob[:, 1])
        else:
            auc = roc_auc_score(
                label_binarize(y_true, classes=list(range(n_classes))),
                y_prob, multi_class="ovr", average="macro",
            )
        return {"accuracy": acc, "f1": f1, "recall": rec, "auc": auc, "n": len(y_true)}

    results = {}

    # ── KNN ──────────────────────────────────────────────────────────────────
    print("  KNN...", end=" ", flush=True)
    knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric="euclidean", n_jobs=-1)
    knn.fit(X["train"], y["train"])
    y_pred = knn.predict(X["test"])
    y_prob = knn.predict_proba(X["test"])
    results["knn"] = _metrics(y["test"], y_pred, y_prob)
    print(f"F1={results['knn']['f1']:.3f}")

    # ── Random Forest ─────────────────────────────────────────────────────────
    print("  Random Forest...", end=" ", flush=True)
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=SEED, n_jobs=-1)
    rf.fit(X["train"], y["train"])
    y_pred = rf.predict(X["test"])
    y_prob = rf.predict_proba(X["test"])
    results["random_forest"] = _metrics(y["test"], y_pred, y_prob)
    print(f"F1={results['random_forest']['f1']:.3f}")

    # ── MLP ───────────────────────────────────────────────────────────────────
    print("  MLP...", end=" ", flush=True)

    # When no val split exists, carve a fraction from train for early stopping
    if len(X["val"]) == 0:
        rng_v   = np.random.default_rng(SEED)
        n_val   = max(1, int(len(X["train"]) * mlp_val_frac))
        val_idx = rng_v.choice(len(X["train"]), size=n_val, replace=False)
        tr_idx  = np.setdiff1d(np.arange(len(X["train"])), val_idx)
        X_tr_mlp = X["train"][tr_idx];  y_tr_mlp = y["train"][tr_idx]
        X_va_mlp = X["train"][val_idx]; y_va_mlp = y["train"][val_idx]
    else:
        X_tr_mlp, y_tr_mlp = X["train"], y["train"]
        X_va_mlp, y_va_mlp = X["val"],   y["val"]

    X_tr_t = torch.from_numpy(X_tr_mlp).float().to(device)
    y_tr_t = torch.from_numpy(y_tr_mlp).long().to(device)
    X_va_t = torch.from_numpy(X_va_mlp).float().to(device)
    y_va_t = torch.from_numpy(y_va_mlp).long().to(device)

    mlp = _MLP(X_tr_t.shape[1], n_classes, hidden=mlp_hidden).to(device)
    opt = torch.optim.Adam(mlp.parameters(), lr=mlp_lr, weight_decay=1e-4)
    rng = np.random.default_rng(SEED)

    best_val, best_state, wait = float("inf"), None, 0
    for epoch in range(mlp_epochs):
        mlp.train()
        perm = rng.permutation(len(X_tr_t))
        for i in range(0, len(X_tr_t), mlp_batch):
            idx = perm[i:i + mlp_batch]
            loss = nn.CrossEntropyLoss()(mlp(X_tr_t[idx]), y_tr_t[idx])
            opt.zero_grad(); loss.backward(); opt.step()

        mlp.eval()
        with torch.no_grad():
            val_loss = nn.CrossEntropyLoss()(mlp(X_va_t), y_va_t).item()
        if val_loss < best_val:
            best_val   = val_loss
            best_state = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
            wait       = 0
        else:
            wait += 1
            if wait >= mlp_patience:
                break

    mlp.load_state_dict(best_state)
    mlp.eval()
    with torch.no_grad():
        X_te_t = torch.from_numpy(X["test"]).float().to(device)
        y_prob  = torch.softmax(mlp(X_te_t), dim=1).cpu().numpy()
        y_pred  = y_prob.argmax(axis=1)
    results["mlp"] = _metrics(y["test"], y_pred, y_prob)
    print(f"F1={results['mlp']['f1']:.3f}")

    return results


In [8]:
_run_dir = Path(CONFIG["checkpoint"]).parent
_clf_dir = _run_dir / "data" / "classifiers"
_feat    = "projections"
_ls_key  = "initial_pure"

suite_results = {"lotss_initial_pure": {}}

for _clf_name, _suite_key in [("rf", "random_forest"), ("knn", "knn"), ("lr", "lr")]:
    _json_path = _clf_dir / f"{_clf_name}_{_ls_key}_{_feat}.json"
    if _json_path.exists():
        with open(_json_path) as _f:
            _d = json.load(_f)
        suite_results["lotss_initial_pure"][_suite_key] = {
            "f1":       _d["f1_macro"],
            "auc":      _d["auc_macro"],
            "accuracy": _d["accuracy"],
            "recall":   _d["recall_macro"],
            "n":        _d["n_test"],
        }
        print(f"  {_clf_name.upper()}: F1={_d['f1_macro']:.3f}  AUC={_d['auc_macro']:.3f}  Acc={_d['accuracy']:.3f}")
    else:
        print(f"  {_clf_name.upper()}: not found ({_json_path})")


  RF: F1=0.604  AUC=0.930  Acc=0.796
  KNN: F1=0.659  AUC=0.913  Acc=0.814
  LR: F1=0.658  AUC=0.920  Acc=0.804


## 6. Supervised Baseline Comparison

Load pre-trained supervised classifier results from `CONFIG["baselines_dir"]`
and compare against all BYOL classifiers on the LoTSS initial_pure test split.


In [9]:
_KNOWN_MODELS = ("enb0", "dualssn", "vit", "cnn", "scatternet", "simplescatternet")

_baselines_dir = Path(CONFIG.get("baselines_dir", "../outputs/supervised_baseline_classifiers"))
baseline_results = {}

if _baselines_dir.exists():
    for _path in sorted(_baselines_dir.rglob("results.json")):
        with open(_path) as _f:
            _res = json.load(_f)
        if _res.get("class_names") != ["FRI", "FRII", "Hybrid", "Spiral", "RelaxedDouble"]:
            continue   # skip non-initial-pure results
        _dir   = _path.parent.name
        _model = next((m for m in _KNOWN_MODELS if m in _dir.lower()), _dir)
        _label_set = _res.get("label_set", "?")
        _key = f"{_model} ({_label_set})" if _label_set != "?" else _model
        baseline_results[_key] = {
            "accuracy": _res["accuracy"],
            "f1":       _res["f1_macro"],
            "auc":      _res["auc_macro"],
        }
    print(f"Loaded {len(baseline_results)} baseline result(s): {list(baseline_results.keys())}")
else:
    print(f"Baselines directory not found: {_baselines_dir}")

# ── Build combined table ───────────────────────────────────────────────────────
_lp  = probe_results.get("lotss_initial_pure", {}).get("test", {})
_suite = suite_results.get("lotss_initial_pure", {})

_byol_classifiers = [
    ("BYOL — Logistic Regression",  _lp),
    ("BYOL — KNN",                  _suite.get("knn",           {})),
    ("BYOL — Random Forest",        _suite.get("random_forest", {})),
    ("BYOL — MLP",                  _suite.get("mlp",           {})),
]

_rows = []
for _name, _r in _byol_classifiers:
    if _r:
        _rows.append({
            "Method":       _name,
            "Accuracy":     _r.get("accuracy", float("nan")),
            "Macro F1":     _r.get("f1",       float("nan")),
            "Macro Recall": _r.get("recall",   float("nan")),
            "Macro AUC":    _r.get("auc",      float("nan")),
        })

for _name, _r in baseline_results.items():
    _rows.append({
        "Method":       f"Supervised — {_name.upper()}",
        "Accuracy":     _r["accuracy"],
        "Macro F1":     _r["f1"],
        "Macro Recall": float("nan"),
        "Macro AUC":    _r["auc"],
    })

df_vs_baselines = pd.DataFrame(_rows).set_index("Method")
print("\n5-class Initial Classification — BYOL vs Supervised Baselines (LoTSS test split)")
display(df_vs_baselines.style
    .format("{:.3f}", na_rep="—")
    .highlight_max(axis=0, props="font-weight:bold"))


Loaded 0 baseline result(s): []

5-class Initial Classification — BYOL vs Supervised Baselines (LoTSS test split)


,Accuracy,Macro F1,Macro Recall,Macro AUC
Method,,,,
BYOL — Logistic Regression,0.804,0.658,0.654,0.920
BYOL — KNN,0.814,0.659,0.592,0.913
BYOL — Random Forest,0.796,0.604,0.521,0.930


## 7. Hyperparameter Sweep Grid

Reads pre-computed classifier results from `data/classifiers/` across all BYOL run
directories and produces sweep line plots. The reference run (from `CONFIG["checkpoint"]`) pins
all hyperparameters except the two chosen axes (`axis_x` / `axis_y` in `GRID_CONFIG`).

Each plot shows metric vs. one sweep variable, with lines coloured by the other variable
and linestyle indicating the downstream classifier (RF / KNN / LR).

In [10]:
GRID_CONFIG = {
    # Two parameters to vary (extracted from run dir names)
    "axis_x":      "f",            # label fraction  (f0, f0.1, f0.5, f1 …)
    "axis_y":      "sw",           # supervision wt  (sw0.01, sw0.1, sw1 …)
    # Which downstream classifier to read
    "classifier":  "rf",           # rf | knn | lr
    "label_set":   "initial_pure",
    "feature_type": "projections",
    # Run search root (parent of all run_* dirs)
    "outputs_root": "../outputs",
    "run_glob":    "enb0_*",
}

In [ ]:
import re as _re
import ipywidgets as widgets

# ── Parseable hyperparameter patterns ─────────────────────────────────────────
_PARAM_RE = {
    "f":        r"_f([\d.]+)_",
    "sw":       r"_sw([\d.]+)_",
    "vicregvar": r"_vicregvar([\d.]+)_",
    "cov":      r"_cov([\d.]+)_",
    "gamma":    r"_gamma([\d.]+)_",
    "ema":      r"_ema([\d.]+)_",
}

def _parse_param(name, param):
    m = _re.search(_PARAM_RE[param], name + "_")
    return float(m.group(1)) if m else None

# ── Infer fixed params from the reference run ──────────────────────────────────
_ref_name  = Path(CONFIG["checkpoint"]).parent.name
_axis_x    = GRID_CONFIG["axis_x"]
_axis_y    = GRID_CONFIG["axis_y"]
_fixed = {
    p: _parse_param(_ref_name, p)
    for p in _PARAM_RE
    if p not in (_axis_x, _axis_y) and _parse_param(_ref_name, p) is not None
}
print("Fixed params:", _fixed)

# ── Scan run directories ────────────────────────────────────────────────────────
_root     = Path(GRID_CONFIG["outputs_root"])
_run_dirs = sorted(_root.glob(GRID_CONFIG["run_glob"]))
_run_dirs = [rd for rd in _run_dirs if _re.search(r"_f[\d.]+_sw[\d.]+", rd.name)]

_CLFS  = ["rf", "knn", "lr"]
_CLF_STYLES = {"rf": "-", "knn": "--", "lr": ":"}
_CLF_LABELS = {"rf": "RF", "knn": "KNN", "lr": "LR"}
_ls    = GRID_CONFIG["label_set"]
_feat  = GRID_CONFIG["feature_type"]

# Build records per classifier
_records = {}
for _clf in _CLFS:
    _records[_clf] = []
    for rd in _run_dirs:
        params = {p: _parse_param(rd.name, p) for p in _PARAM_RE}
        if not all(
            params.get(k) is not None and abs(params[k] - v) < 1e-9
            for k, v in _fixed.items()
        ):
            continue
        json_path = rd / "data" / "classifiers" / f"{_clf}_{_ls}_{_feat}.json"
        if not json_path.exists():
            continue
        with open(json_path) as _fh:
            _d = json.load(_fh)
        _records[_clf].append({
            "x":   params[_axis_x],
            "y":   params[_axis_y],
            "f1":  _d.get("f1_macro",     float("nan")),
            "auc": _d.get("auc_macro",    float("nan")),
            "acc": _d.get("accuracy",     float("nan")),
            "rec": _d.get("recall_macro", float("nan")),
        })
    print(f"  {_clf}: {len(_records[_clf])} run(s) found")

_grids = {_clf: {(r["x"], r["y"]): r for r in recs}
          for _clf, recs in _records.items()}
_all_recs = [r for recs in _records.values() for r in recs]
_xs = sorted(set(r["x"] for r in _all_recs))
_ys = sorted(set(r["y"] for r in _all_recs))

_METRICS   = [("f1", "F1 macro"), ("auc", "AUC macro"),
              ("acc", "Accuracy"), ("rec", "Recall macro")]
_line_cmap = plt.cm.tab10

# ── Plot function ──────────────────────────────────────────────────────────────
def _plot_sweeps(show_isolated=True):
    for sweep_var, group_var, sweep_vals, group_vals, axis_is_x in [
        (_axis_x, _axis_y, _xs, _ys, True),
        (_axis_y, _axis_x, _ys, _xs, False),
    ]:
        if len(sweep_vals) < 2:
            print(f"Skipping {sweep_var} sweep (only {len(sweep_vals)} value(s))")
            continue

        print(f"Sweep: {sweep_var}  |  {_ls}, {_feat}  |  fixed: {_fixed}")

        fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=False)

        for ax, (mkey, mlabel) in zip(axes.flat, _METRICS):
            for gi, gv in enumerate(group_vals):
                color = _line_cmap(gi % 10)
                for _clf in _CLFS:
                    xs_plot, ys_plot = [], []
                    for sv in sweep_vals:
                        key = (sv, gv) if axis_is_x else (gv, sv)
                        rec = _grids[_clf].get(key)
                        val = rec[mkey] if rec is not None else float("nan")
                        if val == val:
                            xs_plot.append(sv)
                            ys_plot.append(val)
                    if not xs_plot:
                        continue
                    if not show_isolated and len(xs_plot) == 1:
                        continue
                    ax.plot(xs_plot, ys_plot,
                            marker="o", markersize=6, linewidth=2.0,
                            linestyle=_CLF_STYLES[_clf],
                            color=color,
                            label=f"{group_var}={gv} / {_CLF_LABELS[_clf]}")

            ax.set_xlabel(sweep_var, fontsize=14)
            ax.set_ylabel(mlabel, fontsize=14)
            ax.set_title(mlabel, fontsize=14)
            ax.tick_params(axis="both", labelsize=13)
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)

        _color_handles = [
            plt.Line2D([0], [0], color=_line_cmap(gi % 10), linewidth=2, label=f"{group_var}={gv}")
            for gi, gv in enumerate(group_vals)
        ]
        _style_handles = [
            plt.Line2D([0], [0], color="grey", linewidth=2,
                       linestyle=_CLF_STYLES[c], label=_CLF_LABELS[c])
            for c in _CLFS
        ]
        fig.legend(
            handles=_color_handles + _style_handles,
            loc="upper center", ncol=len(group_vals) + len(_CLFS),
            fontsize=12, bbox_to_anchor=(0.5, 0),
            frameon=True,
        )

        plt.tight_layout()
        _out = f"{OUT_DIR}/sweep_lines_{_ref_name}_{sweep_var}.png"
        plt.savefig(_out, dpi=150, bbox_inches="tight")
        print(f"Saved → {_out}")
        plt.show()

_toggle = widgets.ToggleButton(
    value=False,
    description="Show isolated dots",
    button_style="",
    icon="circle",
    layout=widgets.Layout(width="200px"),
)
widgets.interact(_plot_sweeps, show_isolated=_toggle)


## 8. Confusion Matrix (BYOL vs Supervised Baseline)

Side-by-side row-normalised confusion matrices for a chosen BYOL classifier and a supervised
baseline. Each cell is split: left half (orange) = BYOL, right half (blue) = supervised.
The BYOL classifier and baseline model are configured in `CM_CONFIG`.

In [12]:
CM_CONFIG = {
    "label_set":       "initial_pure",   # must match key in projs_cache
    "byol_classifier": "knn",            # "random_forest" | "knn" | "linear_probe"
    "baseline_model":  "enb0",           # substring matched against baseline run dir names
    "baseline_dir":    "../outputs/supervised_baseline_classifiers",
    "n_estimators":    200,
    "n_neighbors":     15,
}

In [13]:
_ls  = CM_CONFIG["label_set"]
_key = f"lotss_{_ls}"
_cp  = projs_cache[_key]

y_te_byol = _cp["labels"][_cp["split_ids"] == 2]

_byol_clf  = CM_CONFIG["byol_classifier"]
_run_dir   = Path(CONFIG["checkpoint"]).parent
_clf_dir   = _run_dir / "data" / "classifiers"
_clf_short = {"random_forest": "rf", "knn": "knn", "linear_probe": "lr"}.get(_byol_clf, _byol_clf)
_preds_path = _clf_dir / f"{_clf_short}_{_ls}_projections_test_preds.npy"

if _preds_path.exists():
    y_pred_byol = np.load(_preds_path)
else:
    raise FileNotFoundError(
        f"No pre-computed predictions at {_preds_path}.\n"
        f"Re-run train_byol_classifiers.py with --force on this run to generate them."
    )

print(f"BYOL {_byol_clf}: N_test={len(y_te_byol)}")


BYOL knn: N_test=2005


In [14]:
_bdir = Path(CM_CONFIG["baseline_dir"])
_tag  = CM_CONFIG["baseline_model"]
n_cls = len([ALL_CLASS_NAMES[i] for i in LABEL_SETS[_ls]])

_candidates = [d for d in _bdir.iterdir()
               if d.is_dir() and _tag in d.name.lower()
               and (d / "test_probs.npy").exists()]
if not _candidates:
    raise FileNotFoundError(f"No baseline run found for '{_tag}' in {_bdir}")
_brun = sorted(_candidates, key=lambda d: d.stat().st_mtime)[-1]

_probs_raw  = np.load(_brun / "test_probs.npy")   # (N, n_classes)
_labels_raw = np.load(_brun / "test_labels.npy")  # (N, n_classes) multi-hot

if _probs_raw.shape[1] != n_cls:
    raise ValueError(
        f"Baseline '{_brun.name}' has {_probs_raw.shape[1]} output classes "
        f"but confusion matrix expects {n_cls}. "
        f"Pick a baseline trained on the same label set."
    )

# Apply initial_pure filter: keep only sources with exactly one initial label
_pure_mask  = _labels_raw.sum(axis=1) == 1
_probs_base = _probs_raw[_pure_mask]
_labels_base = _labels_raw[_pure_mask]

y_pred_base = _probs_base.argmax(axis=1)
y_te_base   = _labels_base.argmax(axis=1)
print(f"Baseline {_tag}: {_brun.name}")
print(f"  N_test (all): {len(_probs_raw)}  →  N_test (pure): {len(_probs_base)}")

Baseline enb0: enb0_initial_pure_byolsplit_enb0_20260622_1720
  N_test (all): 2005  →  N_test (pure): 2005


In [ ]:
_class_names = [ALL_CLASS_NAMES[i] for i in LABEL_SETS[_ls]]
n_cls = len(_class_names)

cm_b = confusion_matrix(y_te_byol,  y_pred_byol, labels=list(range(n_cls)))
cm_s = confusion_matrix(y_te_base,  y_pred_base,  labels=list(range(n_cls)))

# Row-normalise (recall per class)
cm_b_n = cm_b / cm_b.sum(axis=1, keepdims=True).clip(min=1)
cm_s_n = cm_s / cm_s.sum(axis=1, keepdims=True).clip(min=1)

# ── Metrics ───────────────────────────────────────────────────────────────────
_byol_json = _clf_dir / f"{_clf_short}_{_ls}_projections.json"
with open(_byol_json) as _f:
    _bj = json.load(_f)
_byol_metrics = {
    "F1":     _bj["f1_macro"],
    "AUC":    _bj["auc_macro"],
    "Acc":    _bj["accuracy"],
    "Recall": _bj["recall_macro"],
}

_base_auc = roc_auc_score(
    label_binarize(y_te_base, classes=list(range(n_cls))),
    _probs_base, multi_class="ovr", average="macro",
)
_base_metrics = {
    "F1":     f1_score(y_te_base, y_pred_base, average="macro", zero_division=0),
    "AUC":    _base_auc,
    "Acc":    accuracy_score(y_te_base, y_pred_base),
    "Recall": recall_score(y_te_base, y_pred_base, average="macro", zero_division=0),
}

print(f"{'Metric':<8}  {'BYOL':>6}  {'Base':>6}")
print("-" * 26)
for k in ["F1", "AUC", "Acc", "Recall"]:
    print(f"{k:<8}  {_byol_metrics[k]:>6.3f}  {_base_metrics[k]:>6.3f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
cmap_b = plt.cm.Oranges
cmap_s = plt.cm.Blues

fig, ax = plt.subplots(figsize=(n_cls * 1.5 + 3, n_cls * 1.5 + 1))
plt.subplots_adjust(left=0.28, right=0.98)
ax.set_xlim(0, n_cls); ax.set_ylim(0, n_cls)
ax.invert_yaxis()

for i in range(n_cls):
    for j in range(n_cls):
        ax.add_patch(plt.Rectangle([j, i],     0.5, 1, color=cmap_b(cm_b_n[i, j])))
        ax.add_patch(plt.Rectangle([j+0.5, i], 0.5, 1, color=cmap_s(cm_s_n[i, j])))
        ax.text(j+0.25, i+0.5, f"{cm_b_n[i,j]:.2f}", ha="center", va="center", fontsize=13)
        ax.text(j+0.75, i+0.5, f"{cm_s_n[i,j]:.2f}", ha="center", va="center", fontsize=13)
        ax.add_patch(plt.Rectangle([j, i], 1, 1, fill=False, edgecolor="grey", lw=0.5))

ax.set_xticks(np.arange(n_cls) + 0.5)
ax.set_xticklabels(_class_names, rotation=45, ha="right", fontsize=16)
ax.set_yticks(np.arange(n_cls) + 0.5)
ax.set_yticklabels(_class_names, fontsize=16)
ax.set_xlabel("Predicted", fontsize=16)
ax.set_ylabel("True", fontsize=16)

# Metrics table — top left of the axes
_metrics_lines = ["       BYOL  Base"]
for k, short in [("F1", "F1"), ("AUC", "AUC"), ("Acc", "Acc"), ("Recall", "Rec")]:
    _metrics_lines.append(f"{short:<4}  {_byol_metrics[k]:.3f} {_base_metrics[k]:.3f}")
fig.text(0.01, 0.95, "\n".join(_metrics_lines),
         va="top", ha="left",
         fontsize=13, family="monospace",
         bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.85, edgecolor="grey"))

fig.legend(handles=[
    mpatches.Patch(color=plt.cm.Oranges(0.7), label=f"BYOL {CM_CONFIG['byol_classifier']}"),
    mpatches.Patch(color=plt.cm.Blues(0.7),   label=f"Supervised {CM_CONFIG['baseline_model']}"),
], loc="lower left", fontsize=14, bbox_to_anchor=(0.01, 0.0), frameon=True)

_run_name = _run_dir.name
plt.savefig(f"{OUT_DIR}/confusion_matrix_{_run_name}_{_ls}.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Label-Fraction Experiment

Train logistic regression probes on subsampled fractions of the train-split
projections. Tests how performance scales with the amount of labelled data,
evaluated on the fixed test split.

In [16]:
# ── Find all BYOL runs with f=0 or sw=0 ───────────────────────────────────────
_outputs_root = Path(GRID_CONFIG["outputs_root"])
_all_runs     = sorted(_outputs_root.glob(GRID_CONFIG["run_glob"]))

_target_runs = [
    rd for rd in _all_runs
    if (_parse_param(rd.name, "f")  == 0.0 or
        _parse_param(rd.name, "sw") == 0.0)
    and (rd / "data" / "byol" / "test_projections.npy").exists()
]
print(f"Found {len(_target_runs)} runs (f=0 or sw=0):")
for rd in _target_runs:
    print(f"  {rd.name}")

# ── Helper: load initial_pure projections from a run dir ──────────────────────
def _load_run_projs(run_dir):
    d = run_dir / "data"
    b = d / "byol"
    X_tr_raw = np.load(b / "labelled_train_projections.npy").astype(np.float32)
    X_te_raw = np.load(b / "test_projections.npy").astype(np.float32)
    # derive splits_dir for label files
    _c = torch.load(run_dir / "byol_model_best.pt", map_location="cpu", weights_only=False)
    _s = run_dir.parent / "data_splits" / str(int(_c["config"]["data_seed"]))
    def _lbl(name):
        p = _s / name
        return np.load(p if p.exists() else d / name)
    lab_path = _s / "labelled_train_labels.npy"
    if not lab_path.exists():
        lab_path = d / "labelled_train_labels.npy"
    if lab_path.exists():
        y_tr_raw = np.load(lab_path)
        if len(y_tr_raw) != len(X_tr_raw):
            # labelled_train_labels.npy was written by a different f_label run.
            # Reconstruct by concatenating labelled + unlabelled labels (covers all of
            # train_idx in order, which matches labelled_train_projections.npy for f=1).
            _unlab = _s / "unlabelled_train_labels.npy"
            if not _unlab.exists():
                _unlab = d / "unlabelled_train_labels.npy"
            if _unlab.exists():
                y_tr_raw = np.concatenate([y_tr_raw, np.load(_unlab)])
            if len(y_tr_raw) != len(X_tr_raw):
                raise ValueError(f"Label count {len(y_tr_raw)} != projection count {len(X_tr_raw)} in {run_dir.name}")
    else:
        y_all = _lbl("train_labels.npy")
        idx   = _lbl("labelled_train_idx.npy")
        y_tr_raw = y_all if len(y_all) == len(X_tr_raw) else y_all[idx]
    y_te_raw = _lbl("test_labels.npy")
    # The shared splits dir may have been overwritten by a run with a different
    # test count (e.g. drop_last). Projections always correspond to the first N
    # test labels, so truncate if the label file is longer.
    if len(y_te_raw) != len(X_te_raw):
        if len(y_te_raw) > len(X_te_raw):
            y_te_raw = y_te_raw[:len(X_te_raw)]
        else:
            raise ValueError(f"test_labels count {len(y_te_raw)} < projection count {len(X_te_raw)} in {run_dir.name}")
    def _pure(y): return y[:, 0:5].sum(axis=1) == 1
    tm, em = _pure(y_tr_raw), _pure(y_te_raw)
    return (X_tr_raw[tm], y_tr_raw[tm][:, 0:5].argmax(1).astype(np.int64),
            X_te_raw[em], y_te_raw[em][:, 0:5].argmax(1).astype(np.int64))

# ── Label-fraction experiment ──────────────────────────────────────────────────
fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
N_CLS = 5   # always initial_pure (5 classes)

_clf_factories = {
    "LogReg": lambda: LogisticRegression(max_iter=1000, random_state=SEED, C=1.0),
    "KNN":    lambda: KNeighborsClassifier(n_neighbors=15),
    "RF":     lambda: RandomForestClassifier(n_estimators=200, random_state=SEED),
}

def _full_proba(clf, X):
    """Return (N, N_CLS) probability matrix, padding zero columns for unseen classes."""
    y_prob = clf.predict_proba(X)
    if y_prob.shape[1] == N_CLS:
        return y_prob
    full = np.zeros((len(X), N_CLS), dtype=np.float64)
    for j, c in enumerate(clf.classes_):
        full[:, int(c)] = y_prob[:, j]
    return full

def _run_label_frac(X_tr, y_tr, X_te, y_te):
    scaler    = StandardScaler().fit(X_tr)
    X_tr_s    = scaler.transform(X_tr)
    X_te_s    = scaler.transform(X_te)
    rng       = np.random.default_rng(SEED)
    results   = {}
    for f in fractions:
        n   = max(4, int(f * len(X_tr_s)))
        idx = rng.choice(len(X_tr_s), size=n, replace=False)
        results[f] = {}
        for clf_name, clf_fn in _clf_factories.items():
            clf    = clf_fn()
            clf.fit(X_tr_s[idx], y_tr[idx])
            y_pred = clf.predict(X_te_s)
            y_prob = _full_proba(clf, X_te_s)
            results[f][clf_name] = {
                "accuracy": accuracy_score(y_te, y_pred),
                "f1":       f1_score(y_te, y_pred, average="macro", zero_division=0),
                "recall":   recall_score(y_te, y_pred, average="macro", zero_division=0),
                "auc":      roc_auc_score(
                                label_binarize(y_te, classes=list(range(N_CLS))),
                                y_prob, multi_class="ovr", average="macro"),
            }
    return results

_CACHE_FILE = "label_fraction_metrics.json"

def _save_metrics(run_dir, results):
    cache_dir = run_dir / "data" / "classifiers"
    cache_dir.mkdir(parents=True, exist_ok=True)
    # JSON requires string keys
    serialisable = {str(f): v for f, v in results.items()}
    with open(cache_dir / _CACHE_FILE, "w") as fp:
        json.dump(serialisable, fp, indent=2)

def _load_metrics(run_dir):
    cache_path = run_dir / "data" / "classifiers" / _CACHE_FILE
    if not cache_path.exists():
        return None
    with open(cache_path) as fp:
        raw = json.load(fp)
    loaded = {float(f): v for f, v in raw.items()}
    # Invalidate cache if it was built with a different set of fractions
    if set(loaded.keys()) != set(fractions):
        return None
    return loaded

all_frac_results = {}   # {run_label: {f: {clf_name: {metric: val}}}}
for rd in _target_runs:
    f_val  = _parse_param(rd.name, "f")
    sw_val = _parse_param(rd.name, "sw")
    label  = f"f={f_val} sw={sw_val}"

    cached = _load_metrics(rd)
    if cached is not None:
        all_frac_results[label] = cached
        print(f"→ {label}  (loaded from cache)")
        continue

    print(f"\n→ {label}  ({rd.name})  — training...")
    X_tr, y_tr, X_te, y_te = _load_run_projs(rd)
    results = _run_label_frac(X_tr, y_tr, X_te, y_te)
    _save_metrics(rd, results)
    all_frac_results[label] = results
    for f in fractions:
        print(f"  frac={f:.2f}  " +
              "  ".join(f"{nm}: acc={results[f][nm]['accuracy']:.3f}"
                        for nm in _clf_factories))

Found 4 runs (f=0 or sw=0):
  enb0_mlp_pd128_clos_lrconst_wd1e-4_lfull_ema0.996_vicregvar5_cov0.05_gamma0.5_f0_sw0.1_20260621_1950
  enb0_mlp_pd256_pond_lrconst_wd1e-4_lfull_ema0.99_vicregvar0_cov0_gamma1.0_f0_sw1_20260608_1409
  enb0_mlp_pd50_pond_lrconst_wd1e-4_lfull_ema0.99_vicregvar0_cov0_gamma1.0_f0_sw1_20260608_1411
  enb0_prepca_pond_lrconst_wd1e-4_lfull_ema0.99_vicregvar0_cov0_gamma1.0_f0_sw1_20260528_1040
→ f=0.0 sw=0.1  (loaded from cache)
→ f=0.0 sw=1.0  (loaded from cache)
→ f=0.0 sw=1.0  (loaded from cache)
→ f=0.0 sw=1.0  (loaded from cache)


In [ ]:
_metrics    = [("accuracy", "Accuracy"), ("f1", "Macro F1"),
               ("recall", "Macro Recall"), ("auc", "Macro AUC (OvR)")]
_clf_styles = {"LogReg": "-", "KNN": "--", "RF": ":"}
_run_labels = list(all_frac_results.keys())
_palette    = plt.cm.tab10(np.linspace(0, 0.9, max(len(_run_labels), 1)))
_run_colors = {lbl: _palette[i] for i, lbl in enumerate(_run_labels)}

xs_pct = [f * 100 for f in fractions]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()

for ax, (metric, ylabel) in zip(axes, _metrics):
    for run_lbl, res in all_frac_results.items():
        color = _run_colors[run_lbl]
        for clf_name, ls in _clf_styles.items():
            ys = [res[f][clf_name][metric] for f in fractions]
            ax.plot(xs_pct, ys, linestyle=ls, color=color, linewidth=1.8,
                    marker="o", markersize=4, label=f"{run_lbl} / {clf_name}")
    ax.set_xlabel("Label fraction (%)", fontsize=18)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.set_title(ylabel, fontsize=24)
    ax.set_xticks(xs_pct)
    ax.set_xticklabels([f"{x:.0f}%" for x in xs_pct], fontsize=18)
    ax.tick_params(axis="y", labelsize=20)
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3)

# Shared legend: one entry per (run, clf) — deduplicated from first subplot
handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc="upper center",
           ncol=len(_clf_styles), fontsize=14,
           bbox_to_anchor=(0.5, 1.01), frameon=True)

plt.suptitle("Label Fraction vs Performance — BYOL runs with f=0 or sw=0",
             fontsize=39, y=1.07)
plt.tight_layout()
_out = f"{OUT_DIR}/label_fraction_{_ref_name}.png"
plt.savefig(_out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {_out}")
